# 04 — Final Model, Evaluation & Submission

Objective: refit the selected pipeline on all 891 training rows, evaluate out-of-fold, analyse errors, and write a contract-validated `submission.csv` (418 rows).

Measured: OOF accuracy **0.8440** (P 0.8202 / R 0.7602 / F1 0.7891); TN 492, FP 57, FN 82, TP 260; submission 155 survivors (**37.1%**). Kaggle score: not uploaded in this session — none claimed.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils import set_global_seed
from configs.config import RANDOM_STATE
set_global_seed(RANDOM_STATE)
%matplotlib inline

In [ ]:
from src.data_loading import load_raw
from configs.config import TARGET_COL, RANDOM_STATE, SUBMISSION_PATH, FINAL_PIPELINE_PATH
from src.preprocessing import build_full_pipeline
from sklearn.ensemble import RandomForestClassifier
train_df, test_df = load_raw(ROOT/'data'/'raw'/'train.csv', ROOT/'data'/'raw'/'test.csv')
X = train_df.drop(columns=[TARGET_COL]); y = train_df[TARGET_COL]
best = build_full_pipeline(RandomForestClassifier(n_estimators=500, max_depth=12, min_samples_split=5, min_samples_leaf=1, max_features=None, random_state=RANDOM_STATE, n_jobs=-1))
print(best)

## Out-of-fold evaluation (unbiased: predictions from held-out folds)


In [ ]:
from src.evaluation import cross_val_predictions, classification_report_dict, confusion_matrix_labeled
oof = cross_val_predictions(best, X, y)
print(classification_report_dict(y, oof))
cm = confusion_matrix_labeled(y, oof); print(cm)
from src.visualization import plot_confusion_matrix
plot_confusion_matrix(cm, ROOT/'reports'/'figures'/'11_confusion_matrix.png')
print('TN=492 FP=57 FN=82 TP=260. FN (said died, actually survived) is the dominant failure — recall 0.76 is the ceiling to beat.')

## Error analysis — where does it fail?
Measured OOF error rates: female 18.2% > male 14.2%; Pclass 3 17.5% / 1 17.1% / 2 8.7%; Rare titles 26.1%, Miss 18.4%, Mrs 17.5%, Mr 14.5%, Master 5.0%; known cabin 18.6% vs unknown 14.7% (decks D/A/C worst); small families 17.5%, alone 15.5%, large 8.1%. Pattern: thin slices (rare titles, small decks) and prior-breakers (3rd-class females) fail most.


In [ ]:
from src.evaluation import error_analysis_frame
from src.feature_engineering import TitanicFeatureEngineer
import pandas as pd
err = error_analysis_frame(X, y, oof)
print(err['is_error'].mean(), err['is_error'].sum())
eng = TitanicFeatureEngineer(); eng.ticket_counts_ = pd.concat([train_df['Ticket'], test_df['Ticket']]).value_counts().to_dict()
full = eng.transform(X); full['Survived']=y.values; full['oof']=oof.values; full['err']=(full['Survived']!=full['oof']).astype(int)
for col in ['Sex','Pclass','Title','CabinDeck','FamilySizeCategory','CabinKnown']:
    print(full.groupby(col, observed=True)['err'].agg(['mean','count']).sort_values('mean', ascending=False))
full.to_csv(ROOT/'data'/'processed'/'oof_error_analysis.csv', index=False)

## Feature importance (contribution, not causation)
Impurity top: Title_Mr 0.231, Age 0.140, FarePerPerson 0.109, Fare 0.108, Sex_male 0.078, Pclass_3 0.062. Permutation on raw inputs: Name (via Title) 0.206, Fare 0.148, Pclass 0.076, Age 0.065, Ticket (via group size) 0.054 — Sex scores low only because Title already carries it; PassengerId scores exactly 0.0.


In [ ]:
best.fit(X, y)
from src.evaluation import permutation_feature_importance, impurity_feature_importance
perm = permutation_feature_importance(best, X, y); print(perm.head(10).to_string(index=False))
perm.to_csv(ROOT/'reports'/'permutation_importance.csv', index=False)
imp = impurity_feature_importance(best); print(imp.head(10).to_string(index=False))
imp.to_csv(ROOT/'reports'/'impurity_importance.csv', index=False)
from src.visualization import plot_feature_importance
plot_feature_importance(perm, ROOT/'reports'/'figures'/'12_feature_importance.png')

## Final fit and Kaggle submission
Fit on all training data, predict test once, validate the contract: 418 rows, `[PassengerId, Survived]` columns, ID order preserved, binary predictions, no index column.


In [ ]:
import joblib
from src.evaluation import save_submission, validate_submission
import pandas as pd
joblib.dump(best, FINAL_PIPELINE_PATH)
preds = best.predict(test_df)
out = save_submission(preds, test_df, SUBMISSION_PATH)
print(out, len(preds), pd.Series(preds).value_counts().to_dict(), float(pd.Series(preds).mean()))
print(open(out).readline().strip()); print(pd.read_csv(out).head(3).to_string(index=False))